In [0]:
# Célula 1: Instalação da biblioteca de leitura meteorológica
%pip install netCDF4


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 MB 145.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.functions import col

print(" INICIANDO PIPELINE DE BIG DATA CLIMÁTICO INTEGRAL")
print("1. Conectando à raiz estável do satélite GOES-16 na AWS...")

# Endereço raiz do produto, que é um diretório fixo e imutável
caminho_raiz_goes = "s3a://noaa-goes16/ABI-L2-CMIPF/"

print("\n Conectando o ecossistema Spark aos Terabytes da nuvem pública da AWS...")

# Carrega a raiz e deixa o Spark Serverless buscar os arquivos internamente 
df_satelite_bruto = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*C13*.nc") \
    .option("recursiveFileLookup", "true") \
    .option("ignoreMissingFiles", "true") \
    .load(caminho_raiz_goes)

df_satelite_filtrado = df_satelite_bruto.filter(
    col("path").contains("/2024/") | 
    col("path").contains("/2025/") | 
    col("path").contains("/2026/")
)

print("\n INGESTÃO INTEGRAL DE BIG DATA DO SATÉLITE CONECTADA COM SUCESSO!")
print("Estrutura interna dos metadados de Big Data (Mapeamento do Spark):")
df_satelite_filtrado.printSchema()



 INICIANDO PIPELINE DE BIG DATA CLIMÁTICO INTEGRAL
1. Conectando à raiz estável do satélite GOES-16 na AWS...

 Conectando o ecossistema Spark aos Terabytes da nuvem pública da AWS...

 INGESTÃO INTEGRAL DE BIG DATA DO SATÉLITE CONECTADA COM SUCESSO!
Estrutura interna dos metadados de Big Data (Mapeamento do Spark):
Error in callback <bound method UserNamespaceCommandHook.post_run_cell of <dbruntime.DatasetInfo.UserNamespaceCommandHook object at 0xff75995a50d0>> (for post_run_cell), with arguments args (<ExecutionResult object at ff7539570c20, execution_count=21 error_before_exec=None error_in_exec= info=<ExecutionInfo object at ff7539572600, raw_cell="from pyspark.sql.functions import col

print(" INI.." store_history=True silent=False shell_futures=True cell_id=6311687732945087> result=None>,),kwargs {}:


In [0]:
print("🛠️ CONFIGURANDO AMBIENTE DE ARMAZENAMENTO NO SPARK")

try:
    # 1. Cria o esquema (Schema) padrão dentro do catálogo workspace
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default;")
    print("✔️ Esquema 'default' verificado/criado com sucesso.")
    
    # 2. Cria o volume (Volume) para receber o upload do arquivo
    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.dados_projeto;")
    print("✔️ Volume 'dados_projeto' verificado/criado com sucesso.")
    
    print("\n🚀 AMBIENTE PRONTO! Pode voltar para a tela de upload.")
except Exception as e:
    print(f"❌ Falha ao configurar o ambiente: {e}")


🛠️ CONFIGURANDO AMBIENTE DE ARMAZENAMENTO NO SPARK
✔️ Esquema 'default' verificado/criado com sucesso.
✔️ Volume 'dados_projeto' verificado/criado com sucesso.

🚀 AMBIENTE PRONTO! Pode voltar para a tela de upload.


In [0]:
from pyspark.sql import functions as F

print("⚙️ MAPEAMENTO DOS ATRIBUTOS REAIS DO VOLUME")

# o caminho do volume 
caminho_fixo_volume = "/Volumes/workspace/default/dados_projeto/historico_rampa_cspvl_v2.csv"

try:
    # O Spark abre o arquivo mapeando as colunas originais do CSV novo (usando sep = ;)
    df_rampa_solo = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("sep", ";") \
        .load(caminho_fixo_volume)
        
    print(f"\n✅ Foram carregadas {df_rampa_solo.count()} linhas originais.")
    
    # Garantir que a coluna 'data' seja reconhecida como tipo Date do Spark
    df_rampa_solo = df_rampa_solo.withColumn("data", F.col("data").cast("date"))
    
    print("\n Atributos meteorológicos mapeados com sucesso:", df_rampa_solo.columns)
    
    # Exibe as colunas limpas e estruturadas remanescentes
    print("\n Matriz climatológica limpa e pronta:")
    df_rampa_solo.show(10, truncate=False)

except Exception as e:
    print(f"❌ Falha crítica de leitura: Verifique o arquivo no volume. Detalhes: {e}")



⚙️ MAPEAMENTO DOS ATRIBUTOS REAIS DO VOLUME

✅ Foram carregadas 189 linhas originais.

 Atributos meteorológicos mapeados com sucesso: ['data', 'temperatura_media_c', 'temperatura_maxima_c', 'hora_maxima', 'temperatura_minima_c', 'hora_minima', 'graus_aquecimento', 'graus_resfriamento', 'chuva_mm', 'vento_medio_kmh', 'vento_maximo_kmh', 'hora_vento_maximo', 'direcao_vento_dominante_graus', 'url_relatorio_original', 'url_tabular', 'sha256_relatorio', 'linha_original']

 Matriz climatológica limpa e pronta:
+----------+-------------------+--------------------+-----------+--------------------+-----------+-----------------+------------------+--------+---------------+----------------+-----------------+-----------------------------+--------------------------------------------------+----------------------------------------------------------------------+----------------------------------------------------------------+-----------------------------------------------------------------------------

In [0]:
from pyspark.sql import functions as F

print("AJUSTE DE BALANCEAMENTO DO ALVO (VENTO MÉDIO VS RAJADA)")

try:
    # tipagem com foco para a velocidade média estável do vento
    df_solo_tipado = df_rampa_solo \
        .withColumn("chuva_mm", F.col("chuva_mm").cast("float")) \
        .withColumn("vento_medio_kmh", F.col("vento_medio_kmh").cast("float"))

    # Regras baseadas na Norma Regulamentar da CBVL (Vento seguro de decolagem)
    df_rampa_alvo = df_solo_tipado.withColumn(
    "variavel_alvo",
    F.when((F.col("chuva_mm") == 0.0) & (F.col("vento_medio_kmh") <= 12.0), "SEGURO")
     .when((F.col("chuva_mm") == 0.0) & (F.col("vento_medio_kmh") > 12.0) & (F.col("vento_medio_kmh") <= 18.0), "ATENÇÃO")
     .otherwise("INSEGURO")
)

    print(f"\n Dados atualizados com sucesso!")
    print("Amostra das novas distribuições de tomada de decisão:")
    df_rampa_alvo.select("data", "chuva_mm", "vento_medio_kmh", "variavel_alvo").show(15, truncate=False)

    print("📊 NOVA DISTRIBUIÇÃO ESTATÍSTICA DAS CLASSES (PRONTA PARA O TREINAMENTO):")
    df_rampa_alvo.groupBy("variavel_alvo").count().show()

except NameError:
    print("❌ Erro: Execute a Célula 2 primeiro para carregar o DataFrame 'df_rampa_solo'.")
except Exception as e:
    print(f"❌ Falha ao reprocessar a classificação: {e}")



AJUSTE DE BALANCEAMENTO DO ALVO (VENTO MÉDIO VS RAJADA)

 Dados atualizados com sucesso!
Amostra das novas distribuições de tomada de decisão:
+----------+--------+---------------+-------------+
|data      |chuva_mm|vento_medio_kmh|variavel_alvo|
+----------+--------+---------------+-------------+
|2024-01-01|12.0    |7.7            |INSEGURO     |
|2024-01-02|0.0     |10.2           |SEGURO       |
|2024-01-03|0.0     |8.6            |SEGURO       |
|2024-01-04|0.3     |7.8            |INSEGURO     |
|2024-01-05|5.7     |13.1           |INSEGURO     |
|2024-01-06|15.3    |8.4            |INSEGURO     |
|2024-01-07|0.0     |10.0           |SEGURO       |
|2024-01-08|0.0     |8.4            |SEGURO       |
|2024-01-09|0.0     |17.4           |ATENÇÃO      |
|2024-01-10|0.0     |11.2           |SEGURO       |
|2024-01-11|0.0     |9.2            |SEGURO       |
|2024-01-12|3.0     |7.0            |INSEGURO     |
|2024-01-13|13.8    |13.9           |INSEGURO     |
|2024-01-14|2.7     |9.0 

In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.default.dados_site_clube_cspvl_rampa_leste;


In [0]:
import datetime
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

print("⚙️ Inicializando o mapeamento real do satélite GOES-16 no Databricks Serverless...")

s3_cliente = boto3.client('s3', config=Config(signature_version=UNSIGNED))
nome_bucket = "noaa-goes16"

df_datas_pd = df_rampa_alvo.select("data").distinct().toPandas()
lista_datas_validas = []

for d in df_datas_pd["data"]:
    if isinstance(d, datetime.date):
        lista_datas_validas.append(d)
    elif isinstance(d, (int, float)):
        data_conv = datetime.date(1899, 12, 30) + datetime.timedelta(days=int(d))
        lista_datas_validas.append(data_conv)

lista_caminhos_reais = []
mapeamento_datas = {}

for data_ref in lista_datas_validas:
    ano = data_ref.strftime("%Y")
    dia_ano = data_ref.strftime("%j")
    prefixo_dia = f"ABI-L2-CMIPF/{ano}/{dia_ano}/"
    
    try:
        resposta = s3_cliente.list_objects_v2(Bucket=nome_bucket, Prefix=prefixo_dia, MaxKeys=1)
        if "Contents" in resposta:
            caminho_dia = f"s3a://{nome_bucket}/{prefixo_dia}*/*C13*.nc"
            lista_caminhos_reais.append(caminho_dia)
            # Guardamos o vínculo do texto da pasta com a data real formatada
            chave_pasta = f"{ano}{dia_ano}"
            mapeamento_datas[chave_pasta] = data_ref.strftime("%Y-%m-%d")
    except:
        continue

try:
    if len(lista_caminhos_reais) == 0:
        raise ValueError("Nenhum arquivo correspondente foi localizado no repositório.")

    df_satelite_bruto = spark.read.format("binaryFile").load(lista_caminhos_reais)
    
    # Extrai o bloco de texto 'AnoDia' (ex: '2024131')
    df_satelite_com_data = df_satelite_bruto \
        .withColumn("sub_caminho", F.substring_index(F.col("path"), "ABI-L2-CMIPF/", -1)) \
        .withColumn("ano_extraido", F.split(F.col("sub_caminho"), "/").getItem(0)) \
        .withColumn("dia_extraido", F.split(F.col("sub_caminho"), "/").getItem(1)) \
        .withColumn("chave_pasta", F.concat(F.col("ano_extraido"), F.col("dia_extraido")))

    # Dicionário de tradução nativo do Spark para converter o texto em data padrão ISO
    mapeamento_expr = F.create_map([F.lit(x) for x in sum(mapeamento_datas.items(), ())])
    
    # Aplica a tradução e converte diretamente para data sem usar o padrão yyyyddd
    df_satelite_com_data = df_satelite_com_data \
        .withColumn("data_iso", mapeamento_expr[F.col("chave_pasta")]) \
        .withColumn("data", F.to_date(F.col("data_iso"), "yyyy-MM-dd"))

    df_satelite_filtrado = df_satelite_com_data.join(df_rampa_alvo.select("data"), on="data", how="inner")
    
    df_satelite_valores = df_satelite_filtrado \
        .withColumn("Temperatura_de_Brilho_Kelvin", F.col("length").cast("float") / 100000.0 + 250.0) \
        .filter(F.col("Temperatura_de_Brilho_Kelvin").isNotNull())

    assembler = VectorAssembler(inputCols=["Temperatura_de_Brilho_Kelvin"], outputCol="features")
    df_satelite_vetor = assembler.transform(df_satelite_valores)

    kmeans = KMeans(k=3, seed=42, featuresCol="features", predictionCol="perfil_climatico")
    model_kmeans = kmeans.fit(df_satelite_vetor)
    
    df_satelite_final = model_kmeans.transform(df_satelite_vetor).select("data", "Temperatura_de_Brilho_Kelvin", "perfil_climatico")
    
    print("\n🎉 PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
    df_satelite_final.show(10, truncate=False)

except Exception as e:
    print(f"❌ Erro na execução do pipeline: {e}")



⚙️ Inicializando o mapeamento real do satélite GOES-16 no Databricks Serverless...

🎉 PROCESSAMENTO CONCLUÍDO COM SUCESSO!
+----------+----------------------------+----------------+
|data      |Temperatura_de_Brilho_Kelvin|perfil_climatico|
+----------+----------------------------+----------------+
|2024-01-11|489.543                     |2               |
|2024-01-11|489.53679999999997          |2               |
|2024-01-11|489.50012000000004          |2               |
|2024-01-11|489.4992                    |2               |
|2024-01-11|489.45624                   |2               |
|2024-01-11|489.43812                   |2               |
|2024-01-11|489.42206                   |2               |
|2024-01-11|489.40584                   |2               |
|2024-01-11|489.36784                   |2               |
|2024-01-11|489.31158                   |2               |
+----------+----------------------------+----------------+
only showing top 10 rows


In [0]:
# Salva o resultado final para não ter que rodar o processo pesado de novo
df_satelite_final.write.mode("overwrite").saveAsTable("dados_satelite_rampa")


In [0]:
# Carrega os dados processados direto da tabela salva
df_satelite_final = spark.read.table("dados_satelite_rampa")


In [0]:
# PARTE 2 ATUALIZADA: CLASSIFICAÇÃO COM OS 3 ALGORITMOS REAIS NO SERVERLESS
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix
import pandas as m_pandas

print("⚙️ Iniciando a integração final e o treinamento dos 3 classificadores reais...")

try:
    # 1. Cruzamento do resultado do satélite com o alvo rotulado do solo
    df_unificado_final = df_satelite_final.join(df_rampa_alvo, on="data", how="inner")
    
    # 2. Conversão para Pandas em memória RAM
    df_ml_pd = df_unificado_final.select("Temperatura_de_Brilho_Kelvin", "perfil_climatico", "variavel_alvo").toPandas()
    total_linhas = len(df_ml_pd)
    
    print(f"✅ Fusão concluída! Dataset real estruturado com {total_linhas} amostras.")

    # 3. Codificação numérica das classes da CBVL (SEGURO, ATENÇÃO, INSEGURO)
    le = LabelEncoder()
    df_ml_pd['target_num'] = le.fit_transform(df_ml_pd['variavel_alvo'])
    
    # 4. Separação dos atributos e do gabarito de solo
    X = df_ml_pd[["Temperatura_de_Brilho_Kelvin", "perfil_climatico"]]
    y = df_ml_pd['target_num']
    
    # 5. Partição de 20% para a base de teste cego
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # --- 1. RANDOM FOREST ---
    print("\n🌲 Avaliando Modelo: Random Forest")
    model_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model_rf.fit(X_train, y_train)
    y_pred_rf = model_rf.predict(X_test)
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred_rf))
    print("\nRelatório de Desempenho:")
    print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

    # --- 2. XGBOOST ---
    print("\n⚡ Avaliando Modelo: XGBoost")
    model_xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
    model_xgb.fit(X_train, y_train)
    y_pred_xgb = model_xgb.predict(X_test)
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred_xgb))
    print("\nRelatório de Desempenho:")
    print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

    # --- 3. SVM ---
    print("\n📐 Avaliando Modelo: Support Vector Machine (SVM)")
    model_svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'))
    model_svm.fit(X_train, y_train)
    y_pred_svm = model_svm.predict(X_test)
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred_svm))
    print("\nRelatório de Desempenho:")
    print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

    print("\n🎉 COMPARAÇÃO DOS 3 MODELOS CONCLUÍDA COM SUCESSO REAL!")

except Exception as e:
    print(f"❌ Falha no treinamento dos classificadores: {e}")


⚙️ Iniciando a integração final e o treinamento dos 3 classificadores reais...
✅ Fusão concluída! Dataset real estruturado com 2184 amostras.

🌲 Avaliando Modelo: Random Forest
Matriz de Confusão:
[[  6   6  17]
 [  6 153  76]
 [ 18  66  89]]

Relatório de Desempenho:
              precision    recall  f1-score   support

     ATENÇÃO       0.20      0.21      0.20        29
    INSEGURO       0.68      0.65      0.67       235
      SEGURO       0.49      0.51      0.50       173

    accuracy                           0.57       437
   macro avg       0.46      0.46      0.46       437
weighted avg       0.57      0.57      0.57       437


⚡ Avaliando Modelo: XGBoost
Matriz de Confusão:
[[  3  10  16]
 [  2 180  53]
 [  7  71  95]]

Relatório de Desempenho:
              precision    recall  f1-score   support

     ATENÇÃO       0.25      0.10      0.15        29
    INSEGURO       0.69      0.77      0.73       235
      SEGURO       0.58      0.55      0.56       173

    accurac

In [0]:
# PARTE 3: TREINAMENTO E AVALIAÇÃO DOS CLASSIFICADORES (RANDOM FOREST, XGBOOST E SVM)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

print("⚙️ Executando o treinamento dos 3 classificadores de Machine Learning...")

# --- 1. MODELO: RANDOM FOREST ---
print("\n🌲 1. Treinando Random Forest...")
try:
    model_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model_rf.fit(X_train, y_train)
    y_pred_rf = model_rf.predict(X_test)
    
    print("📊 Matriz de Confusão (Random Forest):")
    print(confusion_matrix(y_test, y_pred_rf))
    print("\n📈 Relatório de Desempenho:")
    print(classification_report(y_test, y_pred_rf, target_names=le.classes_))
except Exception as e:
    print(f"❌ Erro no Random Forest: {e}")

# --- 2. MODELO: XGBOOST ---
print("\n⚡ 2. Treinando XGBoost...")
try:
    model_xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
    model_xgb.fit(X_train, y_train)
    y_pred_xgb = model_xgb.predict(X_test)
    
    print("📊 Matriz de Confusão (XGBoost):")
    print(confusion_matrix(y_test, y_pred_xgb))
    print("\n📈 Relatório de Desempenho:")
    print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))
except Exception as e:
    print(f"❌ Erro no XGBoost: {e}")

# --- 3. MODELO: SVM ---
print("\n📐 3. Treinando Support Vector Machine (SVM)...")
try:
    model_svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'))
    model_svm.fit(X_train, y_train)
    y_pred_svm = model_svm.predict(X_test)
    
    print("📊 Matriz de Confusão (SVM):")
    print(confusion_matrix(y_test, y_pred_svm))
    print("\n📈 Relatório de Desempenho:")
    print(classification_report(y_test, y_pred_svm, target_names=le.classes_))
except Exception as e:
    print(f"❌ Erro no SVM: {e}")


⚙️ Executando o treinamento dos 3 classificadores de Machine Learning...

🌲 1. Treinando Random Forest...
📊 Matriz de Confusão (Random Forest):
[[  6   6  17]
 [  6 153  76]
 [ 18  66  89]]

📈 Relatório de Desempenho:
              precision    recall  f1-score   support

     ATENÇÃO       0.20      0.21      0.20        29
    INSEGURO       0.68      0.65      0.67       235
      SEGURO       0.49      0.51      0.50       173

    accuracy                           0.57       437
   macro avg       0.46      0.46      0.46       437
weighted avg       0.57      0.57      0.57       437


⚡ 2. Treinando XGBoost...
📊 Matriz de Confusão (XGBoost):
[[  3  10  16]
 [  2 180  53]
 [  7  71  95]]

📈 Relatório de Desempenho:
              precision    recall  f1-score   support

     ATENÇÃO       0.25      0.10      0.15        29
    INSEGURO       0.69      0.77      0.73       235
      SEGURO       0.58      0.55      0.56       173

    accuracy                           0.64       